# KPI 2 — Revenue by Company (Top Buyers + Top-20% Pareto)

**requirements.md (Sales & Revenue, KPI 2):** "Revenue by Company: Top buyer companies by revenue; identify top 20% of customers (Pareto analysis)."

This notebook creates the SQL view `b2b_ecommerce.gold.kpi_2_revenue_by_company` (idempotent: `CREATE OR REPLACE`). Views are catalog objects, not pipeline datasets — this notebook is **not** attached to `silver_to_gold_etl`; run the cell below once (or after any change to the fact/dim definitions).

**Definitions (state formulas):**

* **Grain:** one row per **buying company** (`company_id`) — companies with ≥1 order line with `order_date_valid = true` (0 null `company_sk` in the fact; the 3 companies that only bought on invalid-date orders are excluded, consistent with KPI 1 bases)
* **Gross revenue** = `SUM(line_total)` over ALL statuses with `order_date_valid = true`; **net revenue** = same minus `status = 'cancelled'` (KPI 1 bases, so totals reconcile exactly)
* **Order count** = distinct `order_id` (summed per `company_sk` — every line of an order shares one `company_sk`, verified)
* Aggregation is by `company_id` (business key) via the SCD2 dim join on `company_sk`, so a future second dim version cannot split one buyer into two rows; `company_name`/`country` come from the **current** dim version (`__END_AT IS NULL`)
* **Pareto:** companies ranked by net revenue DESC (`ROW_NUMBER`, tie-break `company_id`); `is_top_20_pct` = rank ≤ `CEIL(0.2 × company_count)`; `cumulative_net_revenue_pct` = running share of total net revenue

**Verified 2026-11-19:** 390 buying companies; net total $28,333,498.48 / gross $30,723,269.53 (exact KPI 1 reconciliation); top 20% = 78 companies holding $22,548,317.85 net = 79.58% of net revenue (gross share 79.53%) — a classic 80/20 profile; all 390 ranks + cumulative % match an independent self-join recompute; 0 null names/countries, 0 orders spanning multiple companies, 13 zero-net companies (all their valid-date orders cancelled — retained on the gross basis, ranked last).

In [0]:
%sql
CREATE OR REPLACE VIEW b2b_ecommerce.gold.kpi_2_revenue_by_company
TBLPROPERTIES (
  'comment' = 'KPI 2 - Revenue by Company (requirements.md: top buyer companies by revenue; identify top 20% of customers, Pareto analysis). Grain: one row per buying company (company_id) with >= 1 order line with order_date_valid = true (KPI 1 bases; 3 companies buying only on invalid-date orders excluded). Gross = SUM(line_total) ALL statuses, valid dates; net = same excluding status = ''cancelled''. Aggregated by company_id (business key) via SCD2 dim join on company_sk so re-versioning cannot split a buyer; name/country from the current dim version (__END_AT IS NULL). Pareto: ROW_NUMBER by net revenue DESC (tie-break company_id); is_top_20_pct = rank <= CEIL(0.2 * company_count); cumulative_net_revenue_pct = 100 * running SUM(net_revenue) / total net revenue, ROUND 2.'
)
AS
WITH per_sk AS (
  SELECT
    f.company_sk,
    SUM(f.line_total)                                                   AS gross_rev,
    SUM(CASE WHEN f.status <> 'cancelled' THEN f.line_total ELSE 0 END) AS net_rev,
    COUNT(DISTINCT f.order_id)                                           AS order_cnt
  FROM b2b_ecommerce.gold.fact_order_items f
  WHERE f.order_date_valid
  GROUP BY f.company_sk
),
per_company AS (
  SELECT
    d.company_id,
    SUM(p.gross_rev) AS gross_revenue,
    SUM(p.net_rev)   AS net_revenue,
    SUM(p.order_cnt)  AS order_count
  FROM per_sk p
  JOIN b2b_ecommerce.gold.dim_companies d ON p.company_sk = d.company_sk
  GROUP BY d.company_id
),
ranked AS (
  SELECT
    c.company_id,
    cd.name    AS company_name,
    cd.country AS country,
    c.gross_revenue,
    c.net_revenue,
    c.order_count,
    ROW_NUMBER() OVER (ORDER BY c.net_revenue DESC, c.company_id) AS net_revenue_rank,
    COUNT(*) OVER ()                       AS n_companies,
    SUM(c.net_revenue) OVER ()             AS total_net
  FROM per_company c
  LEFT JOIN b2b_ecommerce.gold.dim_companies cd
    ON cd.company_id = c.company_id AND cd.__END_AT IS NULL
)
SELECT
  company_id,
  company_name,
  country,
  gross_revenue,
  net_revenue,
  order_count,
  net_revenue_rank,
  ROUND(100 * SUM(net_revenue) OVER (ORDER BY net_revenue_rank ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / total_net, 2) AS cumulative_net_revenue_pct,
  net_revenue_rank <= CEIL(0.2 * n_companies) AS is_top_20_pct
FROM ranked;

In [0]:
%sql
-- Verification: shape, totals, Pareto headline. Expected: 390 rows = 390 distinct company_id,
-- SUM(net) = 28,333,498.48 and SUM(gross) = 30,723,269.53 (exact KPI 1 reconciliation),
-- top-20% = 78 companies holding 22,548,317.85 net (~79.55%), 0 null names/countries,
-- ranks 1..390 contiguous.
SELECT
  COUNT(*)                                                    AS rows_companies,
  COUNT(DISTINCT company_id)                                  AS distinct_companies,
  SUM(net_revenue)                                            AS sum_net,
  SUM(gross_revenue)                                          AS sum_gross,
  SUM(CASE WHEN is_top_20_pct THEN 1 ELSE 0 END)               AS top20_company_count,
  CEIL(0.2 * COUNT(*))                                        AS expected_top20_count,
  SUM(CASE WHEN is_top_20_pct THEN net_revenue ELSE 0 END)     AS top20_net,
  ROUND(100 * SUM(CASE WHEN is_top_20_pct THEN net_revenue ELSE 0 END) / SUM(net_revenue), 2) AS top20_net_share_pct,
  ROUND(100 * SUM(CASE WHEN is_top_20_pct THEN gross_revenue ELSE 0 END) / SUM(gross_revenue), 2) AS top20_gross_share_pct,
  SUM(CASE WHEN company_name IS NULL OR country IS NULL THEN 1 ELSE 0 END) AS null_name_or_country,
  MIN(net_revenue_rank)                                       AS min_rank,
  MAX(net_revenue_rank)                                       AS max_rank,
  COUNT(DISTINCT net_revenue_rank)                             AS distinct_ranks,
  SUM(CASE WHEN net_revenue = 0 THEN 1 ELSE 0 END)             AS zero_net_companies
FROM b2b_ecommerce.gold.kpi_2_revenue_by_company;

In [0]:
%sql
-- Verification: independently recompute EVERY cumulative % via self-join (not window),
-- and confirm rank ordering equals net-revenue-desc ordering. Expected: 390 companies
-- checked, cumulative_match = 390, rank_order_match = 390, flag_match = 390.
WITH v AS (
  SELECT company_id, net_revenue, net_revenue_rank, cumulative_net_revenue_pct, is_top_20_pct,
         SUM(net_revenue) OVER () AS total_net, COUNT(*) OVER () AS n
  FROM b2b_ecommerce.gold.kpi_2_revenue_by_company
),
cum AS (
  SELECT a.company_id,
         ROUND(100 * SUM(b.net_revenue) / a.total_net, 2) AS recompute_cum_pct
  FROM v a
  JOIN v b ON b.net_revenue_rank <= a.net_revenue_rank
  GROUP BY a.company_id, a.total_net
),
ord AS (
  SELECT company_id,
         ROW_NUMBER() OVER (ORDER BY net_revenue DESC, company_id) AS expected_rank
  FROM v
)
SELECT
  COUNT(*) AS companies_checked,
  SUM(CASE WHEN v.cumulative_net_revenue_pct = c.recompute_cum_pct THEN 1 ELSE 0 END) AS cumulative_match,
  SUM(CASE WHEN v.net_revenue_rank = o.expected_rank THEN 1 ELSE 0 END)               AS rank_order_match,
  SUM(CASE WHEN v.is_top_20_pct = (v.net_revenue_rank <= CEIL(0.2 * v.n)) THEN 1 ELSE 0 END) AS flag_match
FROM v
JOIN cum c ON c.company_id = v.company_id
JOIN ord o ON o.company_id = v.company_id;

In [0]:
%sql
-- Top-20% buyer companies (the Pareto segment) with revenue, orders and cumulative share
SELECT * FROM b2b_ecommerce.gold.kpi_2_revenue_by_company
WHERE is_top_20_pct
ORDER BY net_revenue_rank;